# 04 Season Summary Build

## Purpose
Convert scored batted-ball events into a full plate-appearance-level dataset, then aggregate to player-season xwOBA and compare expected versus actual results.

## Inputs
- `data/scored/bbe_scored.parquet`
- `data/raw/statcast_events.parquet`
- `data/raw/woba_weights.parquet`

## Outputs
- `data/final/pa_level_xwoba.parquet`
- `data/final/player_season_xwoba.parquet`

In [1]:
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [2]:
raw_statcast = pd.read_parquet("data/raw/statcast_events.parquet")
bbe_scored = pd.read_parquet("data/scored/bbe_scored.parquet")
woba = pd.read_parquet("data/raw/woba_weights.parquet")

print("Raw Statcast shape:", raw_statcast.shape)
print("Scored BBE shape:", bbe_scored.shape)
print("wOBA weights shape:", woba.shape)
print()
display(raw_statcast.head())
display(bbe_scored.head())
display(woba.head())

Raw Statcast shape: (2143789, 119)
Scored BBE shape: (361509, 31)
wOBA weights shape: (3, 7)



,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season
0,CH,2023-10-01,89.0,-2.8,5.59,"Robertson, Nick",677008,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,9,"Heston Kjerstad grounds out, first baseman Bob...",R,L,R,BAL,BOS,X,3,ground_ball,2,2,2023,-1.53,0.33,0.333018,2.005061,<NA>,<NA>,<NA>,2,9,Bot,158.28,166.83,<NA>,<NA>,<NA>,<NA>,11.122985,-129.176025,-3.49208,-19.471845,26.055263,-27.922064,3.81,1.74,6,96.4,-17,90.7,1703,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.11,0.147,0.152,0.0,1,0,0,2,73,6,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,-0.001,-0.233,66.5,6.5,0.149,0.233,96.4,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.55,1.53,-1.53,31.7,1.676715,-1.896554,41.830979,30.714944,26.41202,2023
1,FF,2023-10-01,96.9,-2.4,5.9,"Robertson, Nick",677008,687798,None,foul,<NA>,<NA>,<NA>,<NA>,5,Foul,R,L,R,BAL,BOS,S,<NA>,None,2,2,2023,-0.76,1.36,0.091181,2.705577,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.558215,-140.741874,-6.148396,-12.116762,34.259201,-12.836434,3.81,1.74,223,78.2,56,98.4,2153,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.13,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,5,4-Seam Fastball,1,6,1,6,6,1,1,6,Infield shade,Standard,211,0.0,0.0,70.8,6.7,<NA>,0.0,88.0,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,1.09,0.76,-0.76,47.4,8.715532,3.692542,40.551342,33.656454,26.020583,2023
2,CH,2023-10-01,90.0,-2.93,5.56,"Robertson, Nick",677008,687798,None,ball,<NA>,<NA>,<NA>,<NA>,13,Ball,R,L,R,BAL,BOS,B,<NA>,None,1,2,2023,-1.65,0.36,-0.24348,0.531787,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,10.328033,-130.515462,-7.371558,-20.978432,27.339657,-26.599717,3.712182,1.775306,<NA>,<NA>,<NA>,91.5,1698,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.14,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,4,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,0.0,0.041,<NA>,<NA>,<NA>,-0.041,<NA>,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.47,1.65,-1.65,30.3,<NA>,<NA>,<NA>,<NA>,<NA>,2023
3,ST,2023-10-01,82.2,-3.09,5.55,"Robertson, Nick",677008,687798,None,ball,<NA>,<NA>,<NA>,<NA>,14,Ball,R,L,R,BAL,BOS,B,<NA>,None,0,2,2023,1.43,0.28,0.808145,0.486074,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6.108013,-119.483805,-5.435467,12.155221,26.646301,-28.491928,3.781975,1.739611,<NA>,<NA>,<NA>,82.4,2786,6.9,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.63,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,3,Sweeper,1,6,1,6,6,1,1,6

,bbe_id,game_date,season,batter_id,batter_name,pitcher,events,outcome_label,launch_speed,launch_angle,sprint_speed,stand,p_throws,balls,strikes,pitch_type,zone,bb_type,hc_x,hc_y,home_team,away_team,gb_prob,ground_ball,sprint_effect,predicted_outcome,prob_1B,prob_2B,prob_3B,prob_HR,prob_OUT
0,716367_73_6_677008_687798,2023-10-01,2023,677008,"Robertson, Nick",687798,field_out,OUT,96.4,-17,27.4,L,R,2,2,CH,9,ground_ball,158.28,166.83,BAL,BOS,9.999998e-01,1,27.4,OUT,0.162701,0.010784,0.000255,1.331716e-06,0.826259
1,716367_71_5_663624_687798,2023-10-01,2023,663624,"Robertson, Nick",687798,field_out,OUT,68.3,46,28.2,R,R,3,1,FF,13,popup,168.39,135.05,BAL,BOS,2.282690e-09,0,0.0,OUT,0.214262,0.060130,0.000059,7.378736e-06,0.725542
2,716367_69_4_624512_608344,2023-10-01,2023,624512,"Irvin, Cole",608344,field_out,OUT,62.6,-64,25.9,L,L,0,2,FC,13,ground_ball,127.46,186.95,BAL,BOS,1.000000e+00,1,25.9,OUT,0.240705,0.000127,0.000005,2.667381e-07,0.759164
3,716367_68_3_622569_608344,2023-10-01,2023,622569,"Irvin, Cole",608344,field_out,OUT,80.5,-38,27.9,R,L,0,2,SI,1,ground_ball,153.72,172.92,BAL,BOS,1.000000e+00,1,27.9,OUT,0.058940,0.000652,0.000015,1.948217e-07,0.940393
4,716367_67_3_623993_670167,2023-10-01,2023,623993,"Schreiber, John",670167,field_out,OUT,82.0,43,26.7,L,R,2,0,ST,5,fly_ball,59.68,124.3,BAL,BOS,2.142430e-09,0,0.0,OUT,0.005989,0.006233,0.000406,3.701069e-06,0.987368


,season,unintentional_bb,hbp,single_weight,double_weight,triple_weight,hr_weight
0,2023,0.696,0.726,0.883,1.244,1.569,2.004
1,2024,0.689,0.720,0.882,1.254,1.590,2.050
2,2025,0.691,0.722,0.882,1.252,1.584,2.037


## Build the regular season plate appearance base

In [3]:
pa_base = raw_statcast.copy()

pa_base = pa_base[pa_base["game_type"] == "R"].copy()
pa_base = pa_base[pa_base["events"].notna()].copy()

for col in ["season", "batter", "pitcher", "game_pk", "at_bat_number", "pitch_number"]:
    if col in pa_base.columns:
        pa_base[col] = pd.to_numeric(pa_base[col], errors="coerce")

if "player_name" in pa_base.columns and "batter_name" not in pa_base.columns:
    pa_base["batter_name"] = pa_base["player_name"]

print("Regular-season PA base shape:", pa_base.shape)
print("Unique seasons:", sorted(pa_base["season"].dropna().astype(int).unique().tolist()))
display(pa_base.head())

Regular-season PA base shape: (550051, 120)
Unique seasons: [2023, 2024, 2025]


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season,batter_name
0,CH,2023-10-01,89.0,-2.8,5.59,"Robertson, Nick",677008,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,9,"Heston Kjerstad grounds out, first baseman Bob...",R,L,R,BAL,BOS,X,3,ground_ball,2,2,2023,-1.53,0.33,0.333018,2.005061,<NA>,<NA>,<NA>,2,9,Bot,158.28,166.83,<NA>,<NA>,<NA>,<NA>,11.122985,-129.176025,-3.49208,-19.471845,26.055263,-27.922064,3.81,1.74,6,96.4,-17,90.7,1703,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.11,0.147,0.152,0.0,1,0,0,2,73,6,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,-0.001,-0.233,66.5,6.5,0.149,0.233,96.4,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.55,1.53,-1.53,31.7,1.676715,-1.896554,41.830979,30.714944,26.41202,2023,"Robertson, Nick"
6,CH,2023-10-01,88.2,-2.86,5.51,"Robertson, Nick",602104,687798,strikeout,swinging_strike,<NA>,<NA>,<NA>,<NA>,13,Ramon Urias strikes out swinging.,R,R,R,BAL,BOS,S,2,None,2,2,2023,-1.45,0.43,-0.815818,1.187745,<NA>,<NA>,<NA>,1,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.123411,-128.123445,-5.373029,-17.605529,26.545899,-26.498222,3.119666,1.459595,<NA>,<NA>,<NA>,89.6,1656,7.2,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.26,<NA>,0.0,0.0,1,0,0,<NA>,72,6,Changeup,1,6,1,6,6,1,1,6,Standard,Standard,254,-0.002,-0.234,66.2,8.5,<NA>,0.234,<NA>,-5,-5,0.003,0.003,24,29,25,29,1,0,11,1,<NA>,6,2.51,1.45,1.45,31.8,12.675671,-25.318306,32.984103,29.051912,38.369551,2023,"Robertson, Nick"
12,FF,2023-10-01,95.8,-2.48,5.88,"Robertson, Nick",663624,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,13,Ryan Mountcastle pops out to second baseman En...,R,R,R,BAL,BOS,X,4,popup,3,1,2023,-0.54,1.58,-1.051937,2.766499,<NA>,<NA>,<NA>,0,9,Bot,168.39,135.05,<NA>,<NA>,<NA>,<NA>,5.071337,-139.246921,-6.233062,-8.158237,33.874278,-10.472415,3.73,1.86,194,68.3,46,97.3,2204,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.13,0.284,0.293,0.0,1,0,0,3,71,5,4-Seam Fastball,1,6,1,6,6,1,1,6,Standard,Standard,212,-0.009,-0.442,69.3,6.4,0.387,0.442,88.0,-5,-5,0.012,0.012,24,26,25,26,1,1,11,1,<NA>,6,0.93,0.54,0.54,42.9,-4.578371,1.280243,40.060882,24.734672,25.749552,2023,"Robertson, Nick"
17,SI,2023-10-01,93.7,1.04,6.37,"Irvin, Cole",678882,608344,strikeout,called_strike,<NA>,<NA>,<NA>,<NA>,1,Ceddanne Rafaela called out on strikes.,R,R,L,BAL,BOS,S,2,None,2,2,2023,1.23,0.85,-0.671859,3.063874,<NA>,<NA>,<NA>,2,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-7.150373,-136.364352,-4.776

## Recreate modeled batted ball events

In [4]:
exclude_events = [
    "catcher_interf",
    "fielders_choice",
    "field_error",
    "fielders_choice_out",
    "sac_fly",
    "sac_fly_double_play",
    "sac_bunt",
    "sac_bunt_double_play"
]
bbe = raw_statcast.copy()
bbe = bbe[bbe["description"] == "hit_into_play"].copy()
bbe = bbe[~bbe["events"].isin(exclude_events)].copy()
bbe = bbe[bbe["game_type"] == "R"].copy()
bbe = bbe[bbe["launch_speed"].notna() & bbe["launch_angle"].notna()].copy()
for col in ["season", "batter", "pitcher", "game_pk", "at_bat_number", "pitch_number"]:
    if col in bbe.columns:
        bbe[col] = pd.to_numeric(bbe[col], errors="coerce")

if "player_name" in bbe.columns and "batter_name" not in bbe.columns:
    bbe["batter_name"] = bbe["player_name"]

bbe["bbe_id"] = (
    bbe["game_pk"].astype("Int64").astype(str) + "_" +
    bbe["at_bat_number"].astype("Int64").astype(str) + "_" +
    bbe["pitch_number"].astype("Int64").astype(str) + "_" +
    bbe["batter"].astype("Int64").astype(str) + "_" +
    bbe["pitcher"].astype("Int64").astype(str)
)
stable_match_rate = bbe["bbe_id"].isin(set(bbe_scored["bbe_id"])).mean()

print("Recreated BBE shape:", bbe.shape)
print(f"Stable bbe_id match rate: {stable_match_rate:.4%}")
print("Duplicate bbe_id count in recreated BBE:", bbe["bbe_id"].duplicated().sum())
display(bbe[["bbe_id", "game_date", "batter", "pitcher", "events"]].head())

bbe_for_merge = bbe.copy()

Recreated BBE shape: (361509, 121)
Stable bbe_id match rate: 100.0000%
Duplicate bbe_id count in recreated BBE: 0


,bbe_id,game_date,batter,pitcher,events
0,716367_73_6_677008_687798,2023-10-01,677008,687798,field_out
12,716367_71_5_663624_687798,2023-10-01,663624,687798,field_out
22,716367_69_4_624512_608344,2023-10-01,624512,608344,field_out
26,716367_68_3_622569_608344,2023-10-01,622569,608344,field_out
29,716367_67_3_623993_670167,2023-10-01,623993,670167,field_out


## Merge probabilities

In [5]:
score_cols = ["bbe_id", "predicted_outcome"] + [c for c in bbe_scored.columns if c.startswith("prob_")]
score_cols = [c for c in score_cols if c in bbe_scored.columns]

bbe_modeled = bbe_for_merge.merge(
    bbe_scored[score_cols].drop_duplicates(subset=["bbe_id"]),
    on="bbe_id",
    how="left"
)

print("Modeled BBE rows:", len(bbe_modeled))
print("Rows missing predicted_outcome after merge:", int(bbe_modeled["predicted_outcome"].isna().sum()))
print()

for col in [c for c in bbe_modeled.columns if c.startswith("prob_")]:
    print(f"{col} missing:", int(bbe_modeled[col].isna().sum()))

display(bbe_modeled.head())

Modeled BBE rows: 361509
Rows missing predicted_outcome after merge: 0

prob_1B missing: 0
prob_2B missing: 0
prob_3B missing: 0
prob_HR missing: 0
prob_OUT missing: 0


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season,batter_name,bbe_id,predicted_outcome,prob_1B,prob_2B,prob_3B,prob_HR,prob_OUT
0,CH,2023-10-01,89.0,-2.8,5.59,"Robertson, Nick",677008,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,9,"Heston Kjerstad grounds out, first baseman Bob...",R,L,R,BAL,BOS,X,3,ground_ball,2,2,2023,-1.53,0.33,0.333018,2.005061,<NA>,<NA>,<NA>,2,9,Bot,158.28,166.83,<NA>,<NA>,<NA>,<NA>,11.122985,-129.176025,-3.49208,-19.471845,26.055263,-27.922064,3.81,1.74,6,96.4,-17,90.7,1703,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.11,0.147,0.152,0.0,1,0,0,2,73,6,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,-0.001,-0.233,66.5,6.5,0.149,0.233,96.4,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.55,1.53,-1.53,31.7,1.676715,-1.896554,41.830979,30.714944,26.41202,2023,"Robertson, Nick",716367_73_6_677008_687798,OUT,0.162701,0.010784,0.000255,1.331716e-06,0.826259
1,FF,2023-10-01,95.8,-2.48,5.88,"Robertson, Nick",663624,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,13,Ryan Mountcastle pops out to second baseman En...,R,R,R,BAL,BOS,X,4,popup,3,1,2023,-0.54,1.58,-1.051937,2.766499,<NA>,<NA>,<NA>,0,9,Bot,168.39,135.05,<NA>,<NA>,<NA>,<NA>,5.071337,-139.246921,-6.233062,-8.158237,33.874278,-10.472415,3.73,1.86,194,68.3,46,97.3,2204,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.13,0.284,0.293,0.0,1,0,0,3,71,5,4-Seam Fastball,1,6,1,6,6,1,1,6,Standard,Standard,212,-0.009,-0.442,69.3,6.4,0.387,0.442,88.0,-5,-5,0.012,0.012,24,26,25,26,1,1,11,1,<NA>,6,0.93,0.54,0.54,42.9,-4.578371,1.280243,40.060882,24.734672,25.749552,2023,"Robertson, Nick",716367_71_5_663624_687798,OUT,0.214262,0.060130,0.000059,7.378736e-06,0.725542
2,FC,2023-10-01,83.6,1.42,6.13,"Irvin, Cole",624512,608344,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,13,"Reese McGuire grounds out, pitcher Cole Irvin ...",R,L,L,BAL,BOS,X,1,ground_ball,0,2,2023,-0.06,-0.17,-0.912408,2.294593,<NA>,<NA>,<NA>,1,9,Top,127.46,186.95,<NA>,<NA>,<NA>,<NA>,-5.249558,-121.736232,-2.093667,0.459916,24.107616,-33.636595,3.42,1.7,1,62.6,-64,83.5,2273,6.2,716367,543510,663624,676059,602104,683002,677008,622761,623993,54.27,0.205,0.183,0.0,1,0,0,2,69,4,Cutter,1,6,6,1,6,1,6,1,Standard,Standard,284,0.001,-0.17,61.8,7.6,0.205,0.17,88.0,-5,5,0.007,0.993,29,28,29,28,1,1,19,1,<NA>,<NA>,3.43,-0.06,-0.06,39.2,0.237634,20.464121,24.866288,47.255717,20.158185,2023,"Irvin, Cole",716367_69_4_624512_608344,OUT,0.240705,0.000127,0.000005,2.667

## Merge modeled BBE results back into the full PA dataset

In [6]:
pa_level = pa_base.copy()
stable_merge_keys = [
    c for c in ["game_date", "game_pk", "at_bat_number", "pitch_number", "batter", "pitcher"]
    if c in pa_level.columns and c in bbe_for_merge.columns
]
pa_level = pa_level.merge(
    bbe_for_merge[stable_merge_keys + ["bbe_id"]].drop_duplicates(),
    on=stable_merge_keys,
    how="left"
)
prob_cols = [c for c in bbe_modeled.columns if c.startswith("prob_")]
merge_back_cols = ["bbe_id", "predicted_outcome"] + prob_cols
pa_level = pa_level.merge(
    bbe_modeled[merge_back_cols].drop_duplicates(subset=["bbe_id"]),
    on="bbe_id",
    how="left"
)

print("PA-level shape after BBE merge:", pa_level.shape)
print("Rows with a matched bbe_id:", int(pa_level["bbe_id"].notna().sum()))
print("Rows with scored probabilities:", int(pa_level[prob_cols[0]].notna().sum()) if len(prob_cols) > 0 else 0)
display(pa_level.head())

PA-level shape after BBE merge: (550051, 127)
Rows with a matched bbe_id: 361509
Rows with scored probabilities: 361509


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season,batter_name,bbe_id,predicted_outcome,prob_1B,prob_2B,prob_3B,prob_HR,prob_OUT
0,CH,2023-10-01,89.0,-2.8,5.59,"Robertson, Nick",677008,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,9,"Heston Kjerstad grounds out, first baseman Bob...",R,L,R,BAL,BOS,X,3,ground_ball,2,2,2023,-1.53,0.33,0.333018,2.005061,<NA>,<NA>,<NA>,2,9,Bot,158.28,166.83,<NA>,<NA>,<NA>,<NA>,11.122985,-129.176025,-3.49208,-19.471845,26.055263,-27.922064,3.81,1.74,6,96.4,-17,90.7,1703,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.11,0.147,0.152,0.0,1,0,0,2,73,6,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,-0.001,-0.233,66.5,6.5,0.149,0.233,96.4,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.55,1.53,-1.53,31.7,1.676715,-1.896554,41.830979,30.714944,26.41202,2023,"Robertson, Nick",716367_73_6_677008_687798,OUT,0.162701,0.010784,0.000255,1.331716e-06,0.826259
1,CH,2023-10-01,88.2,-2.86,5.51,"Robertson, Nick",602104,687798,strikeout,swinging_strike,<NA>,<NA>,<NA>,<NA>,13,Ramon Urias strikes out swinging.,R,R,R,BAL,BOS,S,2,None,2,2,2023,-1.45,0.43,-0.815818,1.187745,<NA>,<NA>,<NA>,1,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.123411,-128.123445,-5.373029,-17.605529,26.545899,-26.498222,3.119666,1.459595,<NA>,<NA>,<NA>,89.6,1656,7.2,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.26,<NA>,0.0,0.0,1,0,0,<NA>,72,6,Changeup,1,6,1,6,6,1,1,6,Standard,Standard,254,-0.002,-0.234,66.2,8.5,<NA>,0.234,<NA>,-5,-5,0.003,0.003,24,29,25,29,1,0,11,1,<NA>,6,2.51,1.45,1.45,31.8,12.675671,-25.318306,32.984103,29.051912,38.369551,2023,"Robertson, Nick",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FF,2023-10-01,95.8,-2.48,5.88,"Robertson, Nick",663624,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,13,Ryan Mountcastle pops out to second baseman En...,R,R,R,BAL,BOS,X,4,popup,3,1,2023,-0.54,1.58,-1.051937,2.766499,<NA>,<NA>,<NA>,0,9,Bot,168.39,135.05,<NA>,<NA>,<NA>,<NA>,5.071337,-139.246921,-6.233062,-8.158237,33.874278,-10.472415,3.73,1.86,194,68.3,46,97.3,2204,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.13,0.284,0.293,0.0,1,0,0,3,71,5,4-Seam Fastball,1,6,1,6,6,1,1,6,Standard,Standard,212,-0.009,-0.442,69.3,6.4,0.387,0.442,88.0,-5,-5,0.012,0.012,24,26,25,26,1,1,11,1,<NA>,6,0.93,0.54,0.54,42.9,-4.578371,1.280243,40.060882,24.734672,25.749552,2023,"Robertson, Nick",716367_71_5_663624_687798,OUT,0.214262,0.060130,0.000059,7.378736e-06,0.725542
3,SI,2023-10-01,93.7,1.04,6.37,"Irvin

## Merge seasonal wOBA weights

In [7]:
weight_cols = [
    "season",
    "unintentional_bb",
    "hbp",
    "single_weight",
    "double_weight",
    "triple_weight",
    "hr_weight"
]
weight_cols = [c for c in weight_cols if c in woba.columns]
pa_level = pa_level.merge(
    woba[weight_cols].drop_duplicates(subset=["season"]),
    on="season",
    how="left"
)

print("Missing seasonal weight counts:")
print(pa_level[[c for c in weight_cols if c != "season"]].isna().sum())
display(pa_level[["season"] + [c for c in weight_cols if c != "season"]].drop_duplicates().sort_values("season"))

Missing seasonal weight counts:
unintentional_bb    0
hbp                 0
single_weight       0
double_weight       0
triple_weight       0
hr_weight           0
dtype: int64


,season,unintentional_bb,hbp,single_weight,double_weight,triple_weight,hr_weight
0,2023,0.696,0.726,0.883,1.244,1.569,2.004
184376,2024,0.689,0.720,0.882,1.254,1.590,2.050
366959,2025,0.691,0.722,0.882,1.252,1.584,2.037


## Compute wOBA/xwOBA value per plate appearance

In [8]:
for col in ["prob_1B", "prob_2B", "prob_3B", "prob_HR"]:
    if col not in pa_level.columns:
        pa_level[col] = np.nan

# Expected wOBA (BBE rows)
pa_level["expected_woba_value_bbe"] = (
    pa_level["prob_1B"].fillna(0.0) * pa_level["single_weight"].fillna(0.0) +
    pa_level["prob_2B"].fillna(0.0) * pa_level["double_weight"].fillna(0.0) +
    pa_level["prob_3B"].fillna(0.0) * pa_level["triple_weight"].fillna(0.0) +
    pa_level["prob_HR"].fillna(0.0) * pa_level["hr_weight"].fillna(0.0)
)

# Expected wOBA (non-BBE rows)
pa_level["expected_woba_value_non_bbe"] = 0.0
pa_level.loc[pa_level["events"] == "walk", "expected_woba_value_non_bbe"] = pa_level["unintentional_bb"]
pa_level.loc[pa_level["events"] == "hit_by_pitch", "expected_woba_value_non_bbe"] = pa_level["hbp"]
pa_level.loc[pa_level["events"] == "intent_walk", "expected_woba_value_non_bbe"] = 0.0

# Combine expected value
pa_level["expected_woba_value"] = np.where(
    pa_level["bbe_id"].notna(),
    pa_level["expected_woba_value_bbe"],
    pa_level["expected_woba_value_non_bbe"]
)

# Actual wOBA (computed, not from column)
pa_level["actual_woba_value"] = 0.0
pa_level.loc[pa_level["events"] == "walk", "actual_woba_value"] = pa_level["unintentional_bb"]
pa_level.loc[pa_level["events"] == "hit_by_pitch", "actual_woba_value"] = pa_level["hbp"]
pa_level.loc[pa_level["events"] == "single", "actual_woba_value"] = pa_level["single_weight"]
pa_level.loc[pa_level["events"] == "double", "actual_woba_value"] = pa_level["double_weight"]
pa_level.loc[pa_level["events"] == "triple", "actual_woba_value"] = pa_level["triple_weight"]
pa_level.loc[pa_level["events"] == "home_run", "actual_woba_value"] = pa_level["hr_weight"]
pa_level.loc[pa_level["events"] == "intent_walk", "actual_woba_value"] = 0.0

value_cols = [
    "events",
    "bbe_id",
    "predicted_outcome",
    "expected_woba_value_bbe",
    "expected_woba_value_non_bbe",
    "expected_woba_value",
    "actual_woba_value"
]

display(pa_level[value_cols].head(10))

,events,bbe_id,predicted_outcome,expected_woba_value_bbe,expected_woba_value_non_bbe,expected_woba_value,actual_woba_value
0,field_out,716367_73_6_677008_687798,OUT,0.157483,0.000,0.157483,0.000
1,strikeout,NaN,NaN,0.000000,0.000,0.000000,0.000
2,field_out,716367_71_5_663624_687798,OUT,0.264102,0.000,0.264102,0.000
3,strikeout,NaN,NaN,0.000000,0.000,0.000000,0.000
4,field_out,716367_69_4_624512_608344,OUT,0.212708,0.000,0.212708,0.000
5,field_out,716367_68_3_622569_608344,OUT,0.052879,0.000,0.052879,0.000
6,field_out,716367_67_3_623993_670167,OUT,0.013686,0.000,0.013686,0.000
7,single,716367_66_4_668939_670167,OUT,0.430482,0.000,0.430482,0.883
8,strikeout,NaN,NaN,0.000000,0.000,0.000000,0.000
9,walk,NaN,NaN,0.000000,0.696,0.696000,0.696


## Build wOBA denominator

In [9]:
pa_level["pa_count"] = 1
pa_level["woba_denom_value"] = pa_level["woba_denom"].fillna(0.0)
pa_level["bb"] = (pa_level["events"] == "walk").astype(int)
pa_level["ibb"] = (pa_level["events"] == "intent_walk").astype(int)
pa_level["hbp_event"] = (pa_level["events"] == "hit_by_pitch").astype(int)
pa_level["sf"] = pa_level["events"].isin(["sac_fly", "sac_fly_double_play"]).astype(int)
pa_level["single"] = (pa_level["events"] == "single").astype(int)
pa_level["double"] = (pa_level["events"] == "double").astype(int)
pa_level["triple"] = (pa_level["events"] == "triple").astype(int)
pa_level["home_run"] = (pa_level["events"] == "home_run").astype(int)
pa_level["hit"] = pa_level[["single", "double", "triple", "home_run"]].sum(axis=1)
ab_exclude = {"walk", "intent_walk", "hit_by_pitch", "sac_fly", "sac_fly_double_play", "sac_bunt", "sac_bunt_double_play", "catcher_interf"}
pa_level["ab"] = (~pa_level["events"].isin(ab_exclude)).astype(int)
summary_cols = [
    "pa_count", "woba_denom_value", "bb", "ibb", "hbp_event", "sf",
    "ab", "hit", "single", "double", "triple", "home_run"
]
print(pa_level[summary_cols].sum().to_frame("total"))

                   total
pa_count          550051
woba_denom_value  545001
bb                 44541
ibb                 1538
hbp_event           6055
sf                  3792
ab                492402
hit               120730
single             77998
double             23730
triple              2036
home_run           16966


In [10]:
group_cols = ["season", "batter"]
if "batter_name" in pa_level.columns:
    group_cols.append("batter_name")
elif "player_name" in pa_level.columns:
    group_cols.append("player_name")
agg_map = {
    "expected_woba_value": "sum",
    "actual_woba_value": "sum",
    "woba_denom_value": "sum",
    "pa_count": "sum",
    "ab": "sum",
    "bb": "sum",
    "ibb": "sum",
    "hbp_event": "sum",
    "sf": "sum",
    "hit": "sum",
    "single": "sum",
    "double": "sum",
    "triple": "sum",
    "home_run": "sum"
}
player_season = (
    pa_level.groupby(group_cols, dropna=False)
    .agg(agg_map)
    .reset_index()
    .rename(columns={
        "expected_woba_value": "expected_numerator",
        "actual_woba_value": "actual_numerator",
        "woba_denom_value": "woba_denom",
        "pa_count": "PA",
        "ab": "AB",
        "bb": "BB",
        "ibb": "IBB",
        "hbp_event": "HBP",
        "sf": "SF",
        "hit": "H",
        "single": "1B",
        "double": "2B",
        "triple": "3B",
        "home_run": "HR"
    })
)
player_season["xwOBA"] = np.where(
    player_season["woba_denom"] > 0,
    player_season["expected_numerator"] / player_season["woba_denom"],
    np.nan
)
player_season["wOBA"] = np.where(
    player_season["woba_denom"] > 0,
    player_season["actual_numerator"] / player_season["woba_denom"],
    np.nan
)
player_season["wOBA_minus_xwOBA"] = player_season["wOBA"] - player_season["xwOBA"]
player_season["xwOBA_minus_wOBA"] = player_season["xwOBA"] - player_season["wOBA"]
player_season = player_season.sort_values(["season", "xwOBA"], ascending=[True, False]).reset_index(drop=True)

print("Player-season summary shape:", player_season.shape)
display(player_season.head(20))

Player-season summary shape: (274713, 21)


,season,batter,batter_name,expected_numerator,actual_numerator,woba_denom,PA,AB,BB,IBB,HBP,SF,H,1B,2B,3B,HR,xwOBA,wOBA,wOBA_minus_xwOBA,xwOBA_minus_wOBA
0,2023,596142,"Chafin, Andrew",2.001590,2.004,1,1,1,0,0,0,0,1,0,0,0,1,2.001590,2.004,0.002410,-0.002410
1,2023,660271,"Nelson, Kyle",2.001590,2.004,1,1,1,0,0,0,0,1,0,0,0,1,2.001590,2.004,0.002410,-0.002410
2,2023,660670,"Hernández, Jonathan",2.001590,2.004,1,1,1,0,0,0,0,1,0,0,0,1,2.001590,2.004,0.002410,-0.002410
3,2023,660670,"Curtiss, John",2.001295,2.004,1,1,1,0,0,0,0,1,0,0,0,1,2.001295,2.004,0.002705,-0.002705
4,2023,683002,"Kowar, Jackson",2.001187,2.004,1,1,1,0,0,0,0,1,0,0,0,1,2.001187,2.004,0.002813,-0.002813
5,2023,592450,"Richards, Trevor",2.000496,2.004,1,1,1,0,0,0,0,1,0,0,0,1,2.000496,2.004,0.003504,-0.003504
6,2023,624413,"Suárez, Andrew",1.999793,2.004,1,1,1,0,0,0,0,1,0,0,0,1,1.999793,2.004,0.004207,-0.004207
7,2023,624413,"Jiménez, Dany",1.999251,2.004,1,1,1,0,0,0,0,1,0,0,0,1,1.999251,2.004,0.004749,-0.004749
8,2023,660271,"Anderson, Grant",1.998698,2.004,1,1,1,0,0,0,0,1,0,0,0,1,1.998698,2.004,0.005302,-0.005302
9,2023,665487,"Marinaccio, Ron",1.998698,2.004,1,1,1,0,0,0,0,1,0,0,0,1,1.998698,2.004,0.005302,-0.005302


## Final validation

In [11]:
print("PA-level rows:", len(pa_level))
print("Modeled BBE rows:", int(pa_level["bbe_id"].notna().sum()))
print("Scored BBE rows with probabilities:", int(pa_level["prob_1B"].notna().sum()))
print()

print("Missing expected_woba_value:", int(pa_level["expected_woba_value"].isna().sum()))
print("Missing actual_woba_value:", int(pa_level["actual_woba_value"].isna().sum()))
print("Missing woba_denom_value:", int(pa_level["woba_denom_value"].isna().sum()))
print()

print("Player-season rows with denominator > 0:", int((player_season["woba_denom"] > 0).sum()))
print("Player-season rows with denominator == 0:", int((player_season["woba_denom"] == 0).sum()))
print()

print("Expected value summary:")
print(pa_level["expected_woba_value"].describe())
print()

print("Actual value summary:")
print(pa_level["actual_woba_value"].describe())

PA-level rows: 550051
Modeled BBE rows: 361509
Scored BBE rows with probabilities: 361509

Missing expected_woba_value: 0
Missing actual_woba_value: 0
Missing woba_denom_value: 0

Player-season rows with denominator > 0: 273106
Player-season rows with denominator == 0: 1607

Expected value summary:
count    550051.000000
mean          0.311109
std           0.377122
min           0.000000
25%           0.000000
50%           0.145319
75%           0.594310
max           2.047519
Name: expected_woba_value, dtype: float64

Actual value summary:
count    550051.000000
mean          0.311493
std           0.511505
min           0.000000
25%           0.000000
50%           0.000000
75%           0.696000
max           2.050000
Name: actual_woba_value, dtype: float64


## Save final outputs

In [12]:
os.makedirs("data/final", exist_ok=True)

pa_level.to_parquet("data/final/pa_level_xwoba.parquet", index=False)
player_season.to_parquet("data/final/player_season_summary.parquet", index=False)

print("Saved: data/final/pa_level_xwoba.parquet")
print("Saved: data/final/player_season_summary.parquet")

Saved: data/final/pa_level_xwoba.parquet
Saved: data/final/player_season_summary.parquet
